# Servidor de generación 3D (Google Colab)

El backend del proyecto (`backend/generate3d.py`) le pide a este servidor un modelo cuando el objeto no está en el catálogo. Corre en Colab porque necesita GPU.

## Cómo usarlo
1. **Entorno de ejecución → Cambiar tipo de entorno → GPU (T4)**.
2. Elegir el generador en la celda de configuración: el que haya ganado en `eval/generacion3d/`.
3. **Entorno de ejecución → Ejecutar todas**. Tarda entre 10 y 15 minutos la primera vez.
4. La última celda imprime tres líneas. Copiarlas al archivo `.env` de la raíz del proyecto y reiniciar el backend.

**Importante**
- La URL cambia cada vez que se corre el notebook, y Colab cierra la sesión después de un rato de inactividad. El día de la defensa hay que correrlo unos 30 minutos antes.
- Los modelos generados quedan guardados en el catálogo del proyecto. Si el servidor se cae, los pedidos ya generados siguen funcionando (plan B del riesgo 1).
- El servidor exige una contraseña (`GEN3D_TOKEN`), porque la URL del túnel es pública.

In [ ]:
import subprocess
import torch

print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), "No hay GPU: Entorno de ejecución → Cambiar tipo de entorno → GPU"

## Configuración

In [ ]:
import secrets

GENERADOR = "triposr"   # "triposr" (texto → imagen → 3D) o "shap-e" (texto → 3D)
PUERTO = 8001
TOKEN = secrets.token_urlsafe(16)
assert GENERADOR in ("triposr", "shap-e")

## Instalación

In [ ]:
!pip install -q diffusers transformers accelerate trimesh fastapi uvicorn
if GENERADOR == "triposr":
    !git clone -q https://github.com/VAST-AI-Research/TripoSR.git /content/TripoSR
    !pip install -q omegaconf einops rembg onnxruntime xatlas
    !pip install -q git+https://github.com/tatsy/torchmcubes.git

## Carga del generador

Mismo código que `eval/generacion3d/evaluar_generadores_colab.ipynb`. SDXL-Turbo usa *CPU offload* para que entre en la T4 junto con TripoSR.

In [ ]:
import inspect
import sys
import time

import numpy as np
import trimesh

if GENERADOR == "triposr":
    import rembg
    from diffusers import AutoPipelineForText2Image
    from PIL import Image

    sys.path.insert(0, "/content/TripoSR")
    from tsr.system import TSR
    from tsr.utils import remove_background, resize_foreground

    sdxl = AutoPipelineForText2Image.from_pretrained("stabilityai/sdxl-turbo", torch_dtype=torch.float16,
                                                     variant="fp16")
    sdxl.enable_model_cpu_offload()
    tsr = TSR.from_pretrained("stabilityai/TripoSR", config_name="config.yaml", weight_name="model.ckpt")
    tsr.renderer.set_chunk_size(8192)
    tsr.to("cuda")
    sesion_rembg = rembg.new_session()
    opciones_malla = {"resolution": 256}
    if "has_vertex_color" in inspect.signature(tsr.extract_mesh).parameters:
        opciones_malla["has_vertex_color"] = True

    def generar_glb(prompt: str) -> bytes:
        imagen = sdxl(prompt=prompt, num_inference_steps=1, guidance_scale=0.0).images[0]
        imagen = resize_foreground(remove_background(imagen, sesion_rembg), 0.85)
        arr = np.array(imagen).astype(np.float32) / 255.0
        arr = arr[:, :, :3] * arr[:, :, 3:4] + (1 - arr[:, :, 3:4]) * 0.5
        imagen = Image.fromarray((arr * 255.0).astype(np.uint8))
        with torch.no_grad():
            codigos = tsr([imagen], device="cuda")
        return tsr.extract_mesh(codigos, **opciones_malla)[0].export(file_type="glb")

else:
    from diffusers import ShapEPipeline
    from diffusers.utils import export_to_ply

    shap_e = ShapEPipeline.from_pretrained("openai/shap-e", torch_dtype=torch.float16, variant="fp16").to("cuda")

    def generar_glb(prompt: str) -> bytes:
        # Shap-E funciona mejor con frases cortas: se usa la primera parte del prompt B
        corto = prompt.split(",")[0]
        salida = shap_e(corto, guidance_scale=15.0, num_inference_steps=64, frame_size=256,
                        output_type="mesh").images[0]
        malla = trimesh.load(export_to_ply(salida, "/content/ultimo.ply"))
        malla.apply_transform(trimesh.transformations.rotation_matrix(-np.pi / 2, [1, 0, 0]))  # Z → Y arriba
        return malla.export(file_type="glb")

print("Generador listo:", GENERADOR)

## Servidor HTTP

In [ ]:
import threading

import uvicorn
from fastapi import FastAPI, Header, HTTPException
from fastapi.responses import Response
from pydantic import BaseModel, Field

app = FastAPI(title="Generador 3D del TPO")
candado = threading.Lock()  # una generación a la vez: la GPU es una sola


class Pedido(BaseModel):
    prompt: str = Field(min_length=1, max_length=400)


@app.get("/salud")
def salud():
    return {"estado": "ok", "generador": GENERADOR}


@app.post("/generar")
def generar(p: Pedido, authorization: str = Header(default="")):
    if not secrets.compare_digest(authorization, f"Bearer {TOKEN}"):
        raise HTTPException(401, "Token inválido")
    with candado:
        t0 = time.perf_counter()
        glb = generar_glb(p.prompt)
        print(f"{p.prompt[:60]!r}: {time.perf_counter() - t0:.1f} s, {len(glb) / 1024:.0f} KB", flush=True)
    return Response(content=glb, media_type="model/gltf-binary")


servidor = uvicorn.Server(uvicorn.Config(app, host="127.0.0.1", port=PUERTO, log_level="warning"))
threading.Thread(target=servidor.run, daemon=True).start()
time.sleep(3)
print("Servidor local en el puerto", PUERTO)

## Prueba local

In [ ]:
import requests

t0 = time.perf_counter()
r = requests.post(f"http://127.0.0.1:{PUERTO}/generar", json={"prompt": "a steel spur gear, centered, plain white background"},
                  headers={"Authorization": f"Bearer {TOKEN}"}, timeout=300)
assert r.status_code == 200 and r.content[:4] == b"glTF", (r.status_code, r.content[:100])
print(f"OK: .glb de {len(r.content) / 1024:.0f} KB en {time.perf_counter() - t0:.1f} s")

## Túnel público (Cloudflare, sin cuenta)

Expone el servidor con una URL `https://….trycloudflare.com` para que el backend pueda llamarlo.

In [ ]:
import re

!wget -q -O /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /content/cloudflared

tunel = subprocess.Popen(["/content/cloudflared", "tunnel", "--url", f"http://127.0.0.1:{PUERTO}"],
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url = None
for linea in tunel.stdout:
    encontrada = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", linea)
    if encontrada:
        url = encontrada.group(0)
        break
assert url, "No se pudo abrir el túnel"

print("Copiá estas líneas al archivo .env del proyecto y reiniciá el backend:\n")
print("GEN3D_MODO=remoto")
print(f"GEN3D_URL={url}")
print(f"GEN3D_TOKEN={TOKEN}")
print("\nDejá esta pestaña abierta mientras se use el visor.")